In [1]:
import pandas as pd

recommendations = pd.read_csv(
    "../data/processed/slot_recommendations_day15.csv"
)

warehouse_slots = pd.read_csv(
    "../data/processed/warehouse_slots.csv"
)

print(recommendations.columns.tolist())
print(warehouse_slots.head())

['StockCode', 'order_frequency', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster_name', 'recommended_zone', 'distance_from_packing', 'picking_priority', 'current_distance', 'distance_saved', 'recommended_distance']
  slot  x  y
0   A1  0  0
1   A2  0  1
2   A3  0  2
3   A4  0  3
4   A5  0  4


In [2]:
def assign_zone(x):
    if x <= 15:
        return "Zone A"
    elif x <= 31:
        return "Zone B"
    elif x <= 47:
        return "Zone C"
    else:
        return "Zone D"

warehouse_slots["zone"] = warehouse_slots["x"].apply(assign_zone)

In [3]:
print(warehouse_slots["zone"].value_counts())

zone
Zone A    25
Name: count, dtype: int64


In [4]:
print(
    warehouse_slots.groupby("zone")["slot"].count()
)

zone
Zone A    25
Name: slot, dtype: int64


In [5]:
optimized_slots = {}

for zone in ["Zone A", "Zone B", "Zone C", "Zone D"]:
    
    available_slots = warehouse_slots[
        warehouse_slots["zone"] == zone
    ]["slot"].tolist()
    
    products = recommendations[
        recommendations["recommended_zone"] == zone
    ]["StockCode"].tolist()
    
    for product, slot in zip(products, available_slots):
        optimized_slots[product] = slot

In [6]:
recommendations["optimized_slot"] = (
    recommendations["StockCode"].map(optimized_slots)
)

In [7]:
print(
    recommendations[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "optimized_slot"
        ]
    ].head(20)
)

   StockCode          cluster_name recommended_zone optimized_slot
0      23309  High-Volume Products           Zone A             A1
1      22086  High-Volume Products           Zone A             A2
2     85123A  High-Volume Products           Zone A             A3
3      22577  High-Volume Products           Zone A             A4
4      84836  High-Volume Products           Zone A             A5
5      22998  High-Volume Products           Zone A             B1
6      22616  High-Volume Products           Zone A             B2
7      15036  High-Volume Products           Zone A             B3
8      22189  High-Volume Products           Zone A             B4
9      22595  High-Volume Products           Zone A             B5
10     17003  High-Volume Products           Zone A             C1
11     71459  High-Volume Products           Zone A             C2
12     84077  High-Volume Products           Zone A             C3
13     20971  High-Volume Products           Zone A           

In [8]:
slot_coordinates = warehouse_slots.set_index("slot")[
    ["x", "y"]
].to_dict("index")

In [9]:
recommendations["optimized_x"] = (
    recommendations["optimized_slot"].map(
        lambda x: slot_coordinates[x]["x"]
        if pd.notna(x) else None
    )
)

recommendations["optimized_y"] = (
    recommendations["optimized_slot"].map(
        lambda x: slot_coordinates[x]["y"]
        if pd.notna(x) else None
    )
)

In [10]:
packing_x = 0
packing_y = 0

In [11]:
recommendations["optimized_distance"] = (
    abs(recommendations["optimized_x"] - packing_x)
    +
    abs(recommendations["optimized_y"] - packing_y)
)

In [12]:
print(
    recommendations[
        [
            "StockCode",
            "current_distance",
            "optimized_distance"
        ]
    ].head(20)
)

   StockCode  current_distance  optimized_distance
0      23309                60                 0.0
1      22086                60                 1.0
2     85123A                60                 2.0
3      22577                60                 3.0
4      84836                60                 4.0
5      22998                60                 1.0
6      22616                60                 2.0
7      15036                60                 3.0
8      22189                60                 4.0
9      22595                60                 5.0
10     17003                60                 2.0
11     71459                60                 3.0
12     84077                60                 4.0
13     20971                60                 5.0
14     21094                60                 6.0
15     21231                60                 3.0
16     23310                60                 4.0
17     20668                60                 5.0
18     22666                60 

In [13]:
recommendations["actual_distance_saved"] = (
    recommendations["current_distance"]
    - recommendations["optimized_distance"]
)

In [14]:
print(
    recommendations[
        [
            "StockCode",
            "current_distance",
            "optimized_distance",
            "actual_distance_saved"
        ]
    ].head(20)
)

   StockCode  current_distance  optimized_distance  actual_distance_saved
0      23309                60                 0.0                   60.0
1      22086                60                 1.0                   59.0
2     85123A                60                 2.0                   58.0
3      22577                60                 3.0                   57.0
4      84836                60                 4.0                   56.0
5      22998                60                 1.0                   59.0
6      22616                60                 2.0                   58.0
7      15036                60                 3.0                   57.0
8      22189                60                 4.0                   56.0
9      22595                60                 5.0                   55.0
10     17003                60                 2.0                   58.0
11     71459                60                 3.0                   57.0
12     84077                60        

In [15]:
recommendations.to_csv(
    "../data/processed/optimized_slot_recommendations.csv",
    index=False
)